# Lab - BUILD: An Agentic System You Can Measure

Three tasks that take the lecture's agent - a gate, a hook, a handoff - and make it a system the business can run: outcomes read from the case facts, the rates stakeholders ask for, and a policy change tested on real cases before anyone decides.

| Task | What you learn |
| --- | --- |
| 1 | An outcome per case, read from the facts the tools recorded |
| 2 | A batch of real messages: automation, escalation and tool calls per case |
| 3 | Change the policy in its config, replay the batch, write the change record |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Use the tools for customers, orders and returns; the tools "
    "check the policy and tell you what is missing. If a tool "
    "refuses and the customer still needs help, hand the case "
    "to a person with escalate. Answer in two sentences."
)

# The business rules live in one place, owned by the business, read by the tools.
POLICY = {"return_window_days": 30}

CUSTOMERS = {"alex@example.com": "CUS-1001"}
ORDERS = {
    "ORD-87654321": {"item": "lamp", "status": "delivered", "delivered_days_ago": 45,
                     "customer_id": "CUS-1001"},
    "ORD-12345678": {"item": "jacket", "status": "delivered", "delivered_days_ago": 20,
                     "customer_id": "CUS-1001"},
}


def schema(field):
    return {"type": "object", "properties": {field: {"type": "string"}}, "required": [field]}


TOOLS = [
    {"name": "get_customer", "description": "Verify a customer by email. Returns the customer ID.",
     "input_schema": schema("email")},
    {"name": "lookup_order", "description": "Look up an order by its ID.",
     "input_schema": schema("order_id")},
    {"name": "start_return", "description": "Start a return for a delivered order inside "
     "the return window.", "input_schema": schema("order_id")},
    {"name": "escalate", "description": "Hand the case to a human agent with a short reason. "
     "Use it when a tool refused and the customer still needs help.",
     "input_schema": schema("reason")},
]

case = {}   # the facts of the case in progress, filled by the tools


def get_customer(email):
    customer_id = CUSTOMERS.get(email)
    if customer_id is None:
        return {"found": False}
    case["customer_id"] = customer_id
    return {"found": True, "customer_id": customer_id}


def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "message": "No order with this ID."}
    case["order_id"] = order_id
    return {"found": True, "order": order}


def start_return(order_id):
    window = POLICY["return_window_days"]
    if ORDERS[order_id]["delivered_days_ago"] > window:
        return {"isError": True, "errorCategory": "business", "isRetryable": False,
                "customerMessage": f"Past the {window}-day return window."}
    case["return_label"] = "RET-2048"
    return {"return_label": "RET-2048"}


def escalate(reason):
    case["escalated"] = True
    return {"ticket": "ESC-507", "reason": reason}


TOOL_FUNCTIONS = {"get_customer": get_customer, "lookup_order": lookup_order,
                  "start_return": start_return, "escalate": escalate}


def before_tool(name, args):
    """The hook from the lecture: runs before every tool call and can block it."""
    if name != "get_customer" and case["customer_id"] is None:
        return {"isError": True, "errorCategory": "permission",
                "customerMessage": "Please give your email first."}
    if name == "start_return" and args["order_id"] != case["order_id"]:
        return {"isError": True, "errorCategory": "validation",
                "customerMessage": "Look up the order first."}
    return None


def run(text):
    """One customer message through the loop; returns the case facts at the end."""
    case.clear()
    case.update({"customer_id": None, "order_id": None, "escalated": False,
                 "return_label": None, "tools": [], "blocked": 0})
    messages = [{"role": "user", "content": text}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=600, output_config={"effort": "low"},
            system=SYSTEM, tools=TOOLS, messages=messages)
        if response.stop_reason != "tool_use":
            break
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = before_tool(block.name, block.input)
                case["blocked"] += output is not None
                if output is None:
                    output = TOOL_FUNCTIONS[block.name](**block.input)
                case["tools"].append(block.name)
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output),
                                "is_error": bool(output.get("isError"))})
        messages.append({"role": "user", "content": results})
    case["answer"] = " ".join(b.text for b in response.content if b.type == "text")
    return dict(case)

---

## Task 1 - One outcome per case, read from the facts

In [ ]:
# ============================================================
# TASK 1 - One outcome per case, read from the facts
# ============================================================
#
# WHAT TO DO
#   Write outcome(): read what happened from the case facts the
#   tools recorded - escalated to a person, waiting for
#   information from the customer, or resolved. Run the
#   lecture's two cases.
#
# WHY IT MATTERS
#   The lecture keeps the case facts in code, not in Claude's
#   words. That is what makes the system measurable: an outcome
#   per case, computed the same way every time, never parsed
#   from a reply that can be phrased a hundred ways.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The case: a small dictionary with the customer ID, the
#   order ID, and whether the case was escalated".
#
# HOW TO DO IT
#   Three blanks:
#
#       "escalated"         a person has it now
#       "customer_id"       None: nobody verified, waiting for the customer
#       "resolved"          a return started, or an order answered
# ============================================================

FIRST = "I want to return the lamp from order ORD-87654321."
SECOND = ("Hi, I am alex@example.com. The lamp from order ORD-87654321 arrived broken "
          "and I want to return it.")

try:
    # BEGIN SOLUTION
    def outcome(facts):
        if facts["escalated"]:
            return "escalated"
        if facts["customer_id"] is None:
            return "needs_info"
        if facts["return_label"] or facts["order_id"]:
            return "resolved"
        return "needs_info"
    # SCAFFOLD: def outcome(facts):
    # SCAFFOLD:     if facts[...]:   # "escalated"
    # SCAFFOLD:         return "escalated"
    # SCAFFOLD:     if facts[...] is None:   # "customer_id"
    # SCAFFOLD:         return "needs_info"
    # SCAFFOLD:     if facts["return_label"] or facts["order_id"]:
    # SCAFFOLD:         return ...   # "resolved"
    # SCAFFOLD:     return "needs_info"
    # END SOLUTION: replace each ... below with the value named beside it

    first, second = run(FIRST), run(SECOND)
    outcomes = [outcome(first), outcome(second)]
    if ... in outcomes:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for facts, result in [(first, outcomes[0]), (second, outcomes[1])]:
    print(f"{result:10} tools {facts['tools']}  blocked {facts['blocked']}")
    print("           ShopAssist:", facts["answer"])

check("system_outcome", outcome=outcome, first=first, second=second)

---

## Task 2 - The numbers stakeholders ask for

In [ ]:
# ============================================================
# TASK 2 - The numbers stakeholders ask for
# ============================================================
#
# WHAT TO DO
#   Run a batch of five real messages and compute the rates:
#   the share resolved by the system alone, the share handed to
#   a person, the share waiting for the customer, and the
#   average number of tool calls per case.
#
# WHY IT MATTERS
#   "Does the agent work?" is not a question a business can act
#   on. The automation rate, the escalation rate and the cost
#   per case are: they size the support team, set the SLA and
#   price the system. The same batch, run again, shows whether a
#   change helped.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The case at the end shows the customer, the order, and
#   escalated true".
#
# HOW TO DO IT
#   Three blanks:
#
#       outcome(facts)                  the outcome of each case
#       count("resolved") / n           the automation rate
#       sum(len(f["tools"]) for f in batch) / n    tool calls per case
# ============================================================

BATCH = [   # provided: five messages from this week's inbox
    FIRST,
    SECOND,
    "Hi, I am alex@example.com. I want to return the jacket from order ORD-12345678.",
    "Hi, I am sam@example.com. I want to return order ORD-12345678.",
    "Hi, I am alex@example.com. Where is my order ORD-12345678?",
]

try:
    # BEGIN SOLUTION
    def kpis(batch):
        n = len(batch)
        results = [outcome(facts) for facts in batch]
        return {"automation_rate": results.count("resolved") / n,
                "escalation_rate": results.count("escalated") / n,
                "needs_info_rate": results.count("needs_info") / n,
                "tool_calls_per_case": sum(len(f["tools"]) for f in batch) / n,
                "outcomes": results}
    # SCAFFOLD: def kpis(batch):
    # SCAFFOLD:     n = len(batch)
    # SCAFFOLD:     results = [... for facts in batch]   # outcome(facts)
    # SCAFFOLD:     return {"automation_rate": ...,   # results.count("resolved") / n
    # SCAFFOLD:             "escalation_rate": results.count("escalated") / n,
    # SCAFFOLD:             "needs_info_rate": results.count("needs_info") / n,
    # SCAFFOLD:             "tool_calls_per_case": ...,   # sum(len(f["tools"]) for f in batch) / n
    # SCAFFOLD:             "outcomes": results}
    # END SOLUTION: replace each ... below with the value named beside it

    batch_30 = [run(text) for text in BATCH]
    kpis_30 = kpis(batch_30)
    if ... in kpis_30.values() or ... in kpis_30["outcomes"]:   # provided: said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
for key, value in kpis_30.items():
    print(f"{key:20} {value if isinstance(value, list) else round(value, 2)}")

check("system_kpis", kpis=kpis_30, batch=batch_30)

---

## Task 3 - Replay the cases before the policy changes

In [ ]:
# ============================================================
# TASK 3 - Replay the cases before the policy changes
# ============================================================
#
# WHAT TO DO
#   The business proposes a 45-day return window. Change the
#   policy value, replay the same batch, and write the change
#   record: the rates before and after, and which cases move.
#   The cell then puts the policy back - the decision is not yours.
#
# WHY IT MATTERS
#   Because the rule lives in one config the tools read, a
#   policy change needs no new prompt and no new code - and it
#   can be tested on real cases before anyone decides. The
#   record tells the business what the change does: which cases
#   leave the human queue, and what that does to the rates.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Give the loop a handoff tool for the cases the policy
#   cannot settle".
#
# HOW TO DO IT
#   Three blanks:
#
#       45                  the proposed window, in days
#       kpis(batch_45)      the same rates, after the change
#       a != b              the outcome changed
# ============================================================

try:
    # BEGIN SOLUTION
    POLICY["return_window_days"] = 45
    try:
        batch_45 = [run(text) for text in BATCH]
    finally:
        POLICY["return_window_days"] = 30   # put it back: the decision is the business's
    kpis_45 = kpis(batch_45)

    moved = [(text, a, b) for text, a, b
             in zip(BATCH, kpis_30["outcomes"], kpis_45["outcomes"]) if a != b]
    # SCAFFOLD: POLICY["return_window_days"] = ...   # 45
    # SCAFFOLD: try:
    # SCAFFOLD:     batch_45 = [run(text) for text in BATCH]
    # SCAFFOLD: finally:
    # SCAFFOLD:     POLICY["return_window_days"] = 30   # put it back: the decision is the business's
    # SCAFFOLD: kpis_45 = ...   # kpis(batch_45)
    # SCAFFOLD:
    # SCAFFOLD: moved = [(text, a, b) for text, a, b
    # SCAFFOLD:          in zip(BATCH, kpis_30["outcomes"], kpis_45["outcomes"]) if ...]   # a != b
    # END SOLUTION: replace each ... below with the value named beside it
    if kpis_45 is ...:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
    change = {   # provided: the record the business decides on
        "change": "return window 30 -> 45 days",
        "automation_rate": f"{kpis_30['automation_rate']:.0%} -> {kpis_45['automation_rate']:.0%}",
        "escalation_rate": f"{kpis_30['escalation_rate']:.0%} -> {kpis_45['escalation_rate']:.0%}",
        "cases_that_move": [f"{a} -> {b}: {text[:50]}" for text, a, b in moved],
    }
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for key, value in change.items():
    print(f"{key:16} {value}")
print("policy now:", POLICY)

check("system_whatif", change=change, moved=moved, policy=POLICY, kpis_45=kpis_45)

---

## Done

Keep the case facts and the business rules in code, measure the system by outcomes read from those facts, and replay real cases before a rule changes - so the business decides with numbers.